In [1]:
import io
import os
from datetime import datetime, timedelta, timezone

import folium
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image

os.makedirs("../images", exist_ok=True)


plt.style.use("bmh")
plt.rcParams["axes.facecolor"] = "white"


yipeng3_df = pd.read_csv("../../data/414270000_20241118_sample(in).csv")
eagle_s_df = pd.read_csv(
    "../../data/EAGLE_S_PositionReport_5min_25Dec2024_26Dec2024.csv"
)
vezhen_df = pd.read_csv("../../data/VEZHEN_PositionReport_5min_25Jan2025_26Jan2025.csv")


all_vessel_df = pd.read_csv(
    "../../data/Vessels_PositionReport_5min_25Dec2024_25Jan2025.csv"
)
all_vessel_near_yipeng = pd.read_parquet(
    "../../data/prepared/baseline/dataset_raw_baseline.parquet"
)


# EAGLE S: take over on 16:40
#
# The `all_vessel_df` has data up to LONGITUDE= only, meaning some data is outside of the window.
# We pick a 0.5 / 2 degree tolerance based on the meteorological data grid.

LON_MAX = all_vessel_df["LONGITUDE"].max() + 0.5
LAT_MAX = all_vessel_df["LATITUDE"].max() + 0.5
LON_MIN = all_vessel_df["LONGITUDE"].min() - 0.5
LAT_MIN = all_vessel_df["LATITUDE"].min() - 0.5

EAGLES_TIME_MIN = datetime(2024, 12, 25, 10, 0, tzinfo=timezone.utc)
EAGLES_TIME_MAX = datetime(2024, 12, 25, 16, 35, tzinfo=timezone.utc)


yipeng3_df.sort_values("static_timestamp", inplace=True)
yipeng3_df.drop_duplicates(subset="static_timestamp", inplace=True)
eagle_s_df.sort_values("DATE TIME (UTC)", inplace=True)
eagle_s_df.drop_duplicates(subset="DATE TIME (UTC)", inplace=True)
vezhen_df.sort_values("DATE TIME (UTC)", inplace=True)
vezhen_df.drop_duplicates(subset="DATE TIME (UTC)", inplace=True)

yipeng3_df["static_timestamp"] = pd.to_datetime(yipeng3_df["static_timestamp"], utc=True)
vezhen_df["DATE TIME (UTC)"] = pd.to_datetime(vezhen_df["DATE TIME (UTC)"], utc=True)
eagle_s_df["DATE TIME (UTC)"] = pd.to_datetime(eagle_s_df["DATE TIME (UTC)"], utc=True)

eagle_s_df = eagle_s_df[
    (
        (
            (eagle_s_df["LATITUDE"].between(LAT_MIN, LAT_MAX))
            & (eagle_s_df["LONGITUDE"].between(LON_MIN, LON_MAX))
        )
        | (eagle_s_df["DATE TIME (UTC)"] > EAGLES_TIME_MIN)
    )
    & (eagle_s_df["DATE TIME (UTC)"] < EAGLES_TIME_MAX)
]

vezhen_df = vezhen_df[
    (vezhen_df["LATITUDE"].between(LAT_MIN, LAT_MAX))
    & (vezhen_df["LONGITUDE"].between(LON_MIN, LON_MAX))
]


# ## Data preparation for ships

yipeng3_even_df = (
    yipeng3_df.set_index("static_timestamp")
    .resample("300s")
    .first()
    .dropna(subset=["mmsi"])
    .reset_index()
)

eagles_even_df = (
    eagle_s_df.set_index("DATE TIME (UTC)")
    .resample("300s")
    .first()
    .dropna(subset=["MMSI"])
    .reset_index()
)

vezhen_even_df = (
    vezhen_df.set_index("DATE TIME (UTC)")
    .resample("300s")
    .first()
    .dropna(subset=["MMSI"])
    .reset_index()
)


print(vezhen_even_df["DATE TIME (UTC)"])


yipeng3_even_df["date_diff"] = yipeng3_even_df["static_timestamp"].diff(1) / timedelta(
    minutes=1
)
eagles_even_df["date_diff"] = eagles_even_df["DATE TIME (UTC)"].diff(1) / timedelta(
    minutes=1
)
vezhen_even_df["date_diff"] = vezhen_even_df["DATE TIME (UTC)"].diff(1) / timedelta(
    minutes=1
)

yipeng3_even_df["dif"] = (
    yipeng3_even_df["heading"] - yipeng3_even_df["course"] + 180
) % 360 - 180
yipeng3_even_df["dif"] = yipeng3_even_df["dif"] / yipeng3_even_df["date_diff"]

eagles_even_df["dif"] = (
    eagles_even_df["HEADING"] - eagles_even_df["COURSE"] + 180
) % 360 - 180
eagles_even_df["dif"] = eagles_even_df["dif"] / eagles_even_df["date_diff"]

vezhen_even_df["dif"] = (
    vezhen_even_df["HEADING"] - vezhen_even_df["COURSE"] + 180
) % 360 - 180
vezhen_even_df["dif"] = vezhen_even_df["dif"] / vezhen_even_df["date_diff"]

yipeng3_even_df["cog_delta"] = (yipeng3_even_df["course"].diff(1) + 180) % 360 - 180
yipeng3_even_df["cog_delta"] = (
    yipeng3_even_df["cog_delta"] / yipeng3_even_df["date_diff"]
)

eagles_even_df["cog_delta"] = (eagles_even_df["COURSE"].diff(1) + 180) % 360 - 180
eagles_even_df["cog_delta"] = eagles_even_df["cog_delta"] / eagles_even_df["date_diff"]

vezhen_even_df["cog_delta"] = (vezhen_even_df["COURSE"].diff(1) + 180) % 360 - 180
vezhen_even_df["cog_delta"] = vezhen_even_df["cog_delta"] / vezhen_even_df["date_diff"]

### Normalize

vezhen_even_df["dif"] = vezhen_even_df["dif"] / 180 * 5
eagles_even_df["dif"] = eagles_even_df["dif"] / 180 * 5
yipeng3_even_df["dif"] = yipeng3_even_df["dif"] / 180 * 5

vezhen_even_df["delta_dif"] = vezhen_even_df["dif"].diff(1) / 2
eagles_even_df["delta_dif"] = eagles_even_df["dif"].diff(1) / 2
yipeng3_even_df["delta_dif"] = yipeng3_even_df["dif"].diff(1) / 2

vezhen_even_df["cog_delta"] = vezhen_even_df["cog_delta"] / 180 * 5
eagles_even_df["cog_delta"] = eagles_even_df["cog_delta"] / 180 * 5
yipeng3_even_df["cog_delta"] = yipeng3_even_df["cog_delta"] / 180 * 5


# ### Filter out incorrect data

eagles_even_df = eagles_even_df[eagles_even_df["SPEED"] <= 25.0]
vezhen_even_df = vezhen_even_df[vezhen_even_df["SPEED"] <= 25.0]
yipeng3_even_df = yipeng3_even_df[yipeng3_even_df["speed"] <= 25.0]


# ### Calculate the moving mean to smooth

for df in [yipeng3_even_df, eagles_even_df, vezhen_even_df]:
    for col in ["cog_delta", "dif", "delta_dif"]:
        df[col] = np.cbrt(df[col]).rolling(3).mean()



0     2025-01-25 00:00:00+00:00
1     2025-01-25 00:05:00+00:00
2     2025-01-25 00:10:00+00:00
3     2025-01-25 00:15:00+00:00
4     2025-01-25 00:20:00+00:00
                 ...           
360   2025-01-26 09:25:00+00:00
361   2025-01-26 09:30:00+00:00
362   2025-01-26 09:35:00+00:00
363   2025-01-26 09:40:00+00:00
364   2025-01-26 09:45:00+00:00
Name: DATE TIME (UTC), Length: 365, dtype: datetime64[ns, UTC]


In [15]:
import warnings
warnings.filterwarnings(action='ignore', category=FutureWarning)
warnings.filterwarnings(action='ignore', category=UserWarning)

In [29]:
fig, axs = plt.subplots(nrows=3, ncols=3, figsize=(12,6), sharex=False, layout='constrained')

ships = ["YI-PENG 3", "EAGLE S", "VEZHEN"]
dfs = [yipeng3_even_df, eagles_even_df, vezhen_even_df]
red_period: list[tuple[datetime, datetime]] = [
    (
        datetime(2024, 11, 18, 8, 10, tzinfo=timezone.utc),
        datetime(2024, 11, 18, 9, 40, tzinfo=timezone.utc),
    ),
    (
        datetime(2024, 12, 25, 11, 40, tzinfo=timezone.utc),
        datetime(2024, 12, 25, 13, 0, tzinfo=timezone.utc),
    ),
    ( 14, 60 ),
]
red_period_names = ["Drifting", "Drifting", "Gale"]
time_column_names = [ "static_timestamp", "DATE TIME (UTC)", "DATE TIME (UTC)" ]
ships_ticks = [
    [ datetime(2024, 11, 18, 0, tzinfo=timezone.utc) + timedelta(hours=i)
      for i in range(3, 24, 10) ],
    [ datetime(2024, 12, 25, 10, 20, tzinfo=timezone.utc) + timedelta(hours=i)
      for i in range(0, 7, 3) ],
    [ datetime(2025, 1, 25, 0, 0, tzinfo=timezone.utc) + timedelta(hours=i)
      for i in range(3, 34, 15) ],
]
target_variables = [
    ("dif", "$DIF$"),
    ("delta_dif", r"$\Delta DIF$"),
    ("cog_delta", r"$\Delta COG$"),
]

for ix, ship in enumerate(ships):
    ship_df = dfs[ix]
    period_start, period_end = red_period[ix]
    red_period_name = red_period_names[ix]
    time_column_name = time_column_names[ix]

    for ix_target, (col_pandas, col_name) in enumerate(target_variables):
        axs[ix_target][ix].plot(
                ship_df[time_column_name],
                ship_df[col_pandas],
                linestyle=(0, (1, 1)),
                label="Anchor dragging and/or\nnormal cruising" if (ix==ix_target)&(ix==2) else None,
            )

        if isinstance(period_start, datetime):
            ship_red = ship_df[
                (ship_df[time_column_name].dt.to_pydatetime() >= period_start)
                & (ship_df[time_column_name].dt.to_pydatetime() < period_end)
            ]
        elif isinstance(period_start, int):
            ship_red = ship_df.iloc[period_start:period_end, :]
        else:
            raise

        axs[ix_target][ix].plot(
            ship_red[time_column_name],
            ship_red[col_pandas],
            color="#ff2222",
            label="Drifting or gale" if (ix==ix_target)&(ix==2) else None,
        )

        ship_labels = [tick.strftime("%Y-%m-%d\n%H:%M") for tick in ships_ticks[ix]]
        axs[ix_target][ix].set_xticks(ticks=ships_ticks[ix], labels=ship_labels)
        
        axs[ix_target][ix].set_ylabel(col_name, size=16)
        axs[ix_target][ix].set_title(f"{col_name} of {ship}")
        axs[ix_target][ix].legend().remove()

for ax in axs.ravel():
    ax.tick_params(axis="both", labelsize=12)

fig.legend(loc='outside lower center', ncol=2, fontsize=14, bbox_to_anchor=(0.5, -0.1))

plt.tight_layout()
plt.savefig(
    "../images/Figure_2.tif",
    format="tiff",
    dpi=1000,
    pil_kwargs={"compression": "tiff_lzw"},
    bbox_inches='tight'
)
plt.close()